# NB10 · ResNet-50 frente a YOLO11 y YOLO26 — palma africana

**Proyecto:** PALMA_AFRICANA_YOLO  
**Caso del libro:** clasificación de madurez y condición del fruto  
**Arquitectura adicional:** ResNet-50 preentrenada en ImageNet  
**Entorno recomendado para entrenamiento:** GPU L4  
**Tiempo estimado:** L4, 15–25 min; T4, 25–40 min  
**CPU:** posible, pero no recomendada para entrenamiento (aprox. 2–4 h)

Este notebook:

- requiere que NB09 haya terminado;
- entrena ResNet-50 con semillas 2026, 2027 y 2028;
- selecciona cada corrida mediante F1 macro de validación;
- evalúa una sola vez en prueba;
- integra ResNet-50 con las 18 corridas YOLO11/YOLO26;
- genera figuras editoriales en PNG a 600 dpi y PDF vectorial para el capítulo 7.


In [ ]:
%pip install -q scikit-learn seaborn SciencePlots adjustText


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import gc
import json
import platform
import random
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torchvision
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
)
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import ResNet50_Weights, resnet50

if not torch.cuda.is_available():
    raise RuntimeError(
        'No se detectó GPU. En Colab seleccione Entorno de ejecución > '
        'Cambiar tipo de entorno de ejecución > GPU L4 o T4.'
    )

DEVICE = torch.device('cuda:0')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB)')
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)


## 1. Rutas y protocolo


In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
DRIVE_DATA = PROJECT_ROOT / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
MODELS_ROOT = PROJECT_ROOT / '05_MODELOS'
RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS'
NB09_RESULTS = RESULTS_ROOT / 'NB09_BENCHMARK_YOLO11_YOLO26_MULTISEED'
NB10_RESULTS = RESULTS_ROOT / 'NB10_RESNET50_BENCHMARK'

LOCAL_DATA = Path('/content/PALMA_CLS_NB10')
CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
SEEDS = [2026, 2027, 2028]
EPOCHS = 30
PATIENCE = 7
BATCH = 32
IMAGE_SIZE = 224

for path in [MODELS_ROOT, NB10_RESULTS]:
    path.mkdir(parents=True, exist_ok=True)

required = [
    DRIVE_DATA,
    NB09_RESULTS / 'benchmark_YOLO11_YOLO26_por_corrida.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Faltan resultados de NB09 o el dataset. Ejecute primero NB09:\n' + '\n'.join(missing)
    )

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(DRIVE_DATA, LOCAL_DATA)
print('Datos copiados al disco local.')


## 2. Transformaciones y cargadores reproducibles


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.15, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loaders(seed):
    train_set = datasets.ImageFolder(LOCAL_DATA / 'train', transform=train_transform)
    val_set = datasets.ImageFolder(LOCAL_DATA / 'val', transform=eval_transform)
    test_set = datasets.ImageFolder(LOCAL_DATA / 'test', transform=eval_transform)
    if train_set.classes != CLASSES:
        raise RuntimeError(f'Orden inesperado de clases: {train_set.classes}')
    generator = torch.Generator().manual_seed(seed)
    common = dict(
        batch_size=BATCH, num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, persistent_workers=True,
    )
    train_loader = DataLoader(train_set, shuffle=True, generator=generator, **common)
    val_loader = DataLoader(val_set, shuffle=False, **common)
    test_loader = DataLoader(test_set, shuffle=False, **common)
    return train_set, val_set, test_set, train_loader, val_loader, test_loader


train_set, val_set, test_set, *_ = make_loaders(SEEDS[0])
assert (len(train_set), len(val_set), len(test_set)) == (312, 70, 70)
print('Clases:', train_set.classes)
print('Train/val/test:', len(train_set), len(val_set), len(test_set))


## 3. Funciones de entrenamiento y evaluación


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_model():
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    return model.to(DEVICE)


def class_weights_from_dataset(dataset):
    targets = np.asarray(dataset.targets)
    counts = np.bincount(targets, minlength=len(CLASSES))
    weights = len(targets) / (len(CLASSES) * counts)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def evaluate_classifier(model, loader, criterion=None):
    model.eval()
    true_labels, pred_labels, probabilities = [], [], []
    losses = []
    start = time.perf_counter()
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(images)
            if criterion is not None:
                losses.append(float(criterion(logits, labels).cpu()))
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            true_labels.extend(labels.cpu().numpy().tolist())
            pred_labels.extend(preds.cpu().numpy().tolist())
            probabilities.extend(probs.cpu().numpy().tolist())
    elapsed = time.perf_counter() - start
    y_true = np.asarray(true_labels)
    y_pred = np.asarray(pred_labels)
    report = classification_report(
        y_true, y_pred, labels=list(range(len(CLASSES))), target_names=CLASSES,
        output_dict=True, zero_division=0,
    )
    metrics = {
        'loss': float(np.mean(losses)) if losses else np.nan,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'inference_ms_image': float(elapsed * 1000 / len(y_true)),
    }
    for label in CLASSES:
        metrics[f'f1_{label}'] = float(report[label]['f1-score'])
    return metrics, y_true, y_pred, np.asarray(probabilities)


## 4. Tres corridas de ResNet-50


In [ ]:
def run_seed(seed):
    run_name = f'resnet50_seed{seed}'
    run_dir = NB10_RESULTS / run_name
    model_dir = MODELS_ROOT / run_name
    summary_path = run_dir / 'resumen.json'
    best_path = model_dir / 'best.pt'
    run_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)

    if summary_path.exists() and best_path.exists():
        print('REUTILIZADA:', run_name)
        return json.loads(summary_path.read_text(encoding='utf-8'))

    set_seed(seed)
    train_set, _, _, train_loader, val_loader, test_loader = make_loaders(seed)
    model = build_model()
    criterion = nn.CrossEntropyLoss(weight=class_weights_from_dataset(train_set))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=True)

    best_val_f1 = -1.0
    epochs_without_improvement = 0
    history = []
    start_train = time.time()

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []
        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=True):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.detach().cpu()))
        scheduler.step()

        val_metrics, _, _, _ = evaluate_classifier(model, val_loader, criterion)
        row = {
            'epoch': epoch + 1,
            'train_loss': float(np.mean(train_losses)),
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_f1_macro': val_metrics['f1_macro'],
            'learning_rate': float(optimizer.param_groups[0]['lr']),
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / 'historial.csv', index=False)
        print(run_name, row)

        if val_metrics['f1_macro'] > best_val_f1 + 1e-4:
            best_val_f1 = val_metrics['f1_macro']
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_path)
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Parada temprana:', run_name)
            break

    training_minutes = (time.time() - start_train) / 60
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    val_metrics, val_true, val_pred, _ = evaluate_classifier(model, val_loader, criterion)
    test_metrics, test_true, test_pred, test_probs = evaluate_classifier(model, test_loader, criterion)

    pd.DataFrame(
        confusion_matrix(test_true, test_pred), index=CLASSES, columns=CLASSES,
    ).to_csv(run_dir / 'matriz_confusion_test.csv')
    pred_frame = pd.DataFrame({
        'true_class': [CLASSES[i] for i in test_true],
        'pred_class': [CLASSES[i] for i in test_pred],
    })
    for class_id, label in enumerate(CLASSES):
        pred_frame[f'prob_{label}'] = test_probs[:, class_id]
    pred_frame.to_csv(run_dir / 'predicciones_test.csv', index=False)

    parameters = sum(p.numel() for p in model.parameters())
    summary = {
        'run_name': run_name,
        'family': 'ResNet',
        'model_key': 'resnet50',
        'architecture': 'ResNet50',
        'seed': seed,
        'training_minutes': float(training_minutes),
        'model_size_mb': float(best_path.stat().st_size / 1024**2),
        'parameters_millions': float(parameters / 1e6),
        **{f'val_{key}': value for key, value in val_metrics.items()},
        **{f'test_{key}': value for key, value in test_metrics.items()},
    }
    summary_path.write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(
        f"FINALIZADA: {run_name} | val F1={summary['val_f1_macro']:.4f} "
        f"| test F1={summary['test_f1_macro']:.4f}"
    )
    del model
    torch.cuda.empty_cache()
    gc.collect()
    return summary


resnet_summaries = []
for position, seed in enumerate(SEEDS, start=1):
    print(f'\n=== ResNet-50 corrida {position}/{len(SEEDS)} ===')
    resnet_summaries.append(run_seed(seed))
    pd.DataFrame(resnet_summaries).to_csv(
        NB10_RESULTS / 'resultados_ResNet50_parciales.csv', index=False
    )

resnet_runs = pd.DataFrame(resnet_summaries).sort_values('seed').reset_index(drop=True)
resnet_runs.to_csv(NB10_RESULTS / 'resultados_ResNet50_por_corrida.csv', index=False)
display(resnet_runs[[
    'seed', 'val_accuracy', 'val_f1_macro', 'test_accuracy', 'test_f1_macro',
    'training_minutes', 'model_size_mb', 'test_inference_ms_image',
]].round(4))


## 5. Integración con YOLO11 y YOLO26


In [ ]:
yolo_runs = pd.read_csv(NB09_RESULTS / 'benchmark_YOLO11_YOLO26_por_corrida.csv')
if 'architecture' not in yolo_runs.columns:
    yolo_runs['architecture'] = yolo_runs['family'] + yolo_runs['size']

required_columns = [
    'architecture', 'family', 'seed', 'val_accuracy', 'val_f1_macro',
    'test_accuracy', 'test_f1_macro', 'training_minutes',
    'model_size_mb', 'test_inference_ms_image',
] + [f'test_f1_{label}' for label in CLASSES]

all_runs = pd.concat([
    yolo_runs[required_columns],
    resnet_runs[required_columns],
], ignore_index=True)
all_runs.to_csv(NB10_RESULTS / 'benchmark_YOLO_ResNet50_por_corrida.csv', index=False)

aggregate = all_runs.groupby(['family', 'architecture']).agg(
    val_accuracy_mean=('val_accuracy', 'mean'),
    val_accuracy_std=('val_accuracy', 'std'),
    val_f1_macro_mean=('val_f1_macro', 'mean'),
    val_f1_macro_std=('val_f1_macro', 'std'),
    test_accuracy_mean=('test_accuracy', 'mean'),
    test_accuracy_std=('test_accuracy', 'std'),
    test_f1_macro_mean=('test_f1_macro', 'mean'),
    test_f1_macro_std=('test_f1_macro', 'std'),
    training_minutes_mean=('training_minutes', 'mean'),
    model_size_mb_mean=('model_size_mb', 'mean'),
    inference_ms_image_mean=('test_inference_ms_image', 'mean'),
).reset_index()
aggregate = aggregate.sort_values('val_f1_macro_mean', ascending=False).reset_index(drop=True)
aggregate.insert(0, 'validation_rank', np.arange(1, len(aggregate) + 1))
aggregate.to_csv(NB10_RESULTS / 'benchmark_resumen_por_modelo.csv', index=False)
display(aggregate.round(4))


## 6. Figuras editoriales para el capítulo 7

La siguiente celda reutiliza los resultados ya guardados y **no repite los entrenamientos**. Puede ejecutarse con CPU. Produce una versión PNG a 600 dpi y otra PDF vectorial de cada figura.


In [ ]:
# FIGURAS EDITORIALES PARA EL CAPÍTULO 7
# Esta celda NO entrena modelos: reutiliza los CSV generados por NB10.
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
import seaborn as sns
import scienceplots  # registra los estilos editoriales
from adjustText import adjust_text

PALETTE = {
    "YOLO11": "#0072B2",
    "YOLO26": "#D55E00",
    "ResNet": "#009E73",
}
MARKERS = {"YOLO11": "o", "YOLO26": "s", "ResNet": "D"}
SELECTED = "YOLO26s"
HIGHLIGHT = "#F0E442"

def decimal_comma(decimals: int = 2):
    return FuncFormatter(lambda value, _: f"{value:.{decimals}f}".replace(".", ","))


def configure_style() -> None:
    plt.style.use(["science", "no-latex"])
    sns.set_theme(style="ticks", context="paper")
    plt.rcParams.update(
        {
            "font.family": "DejaVu Serif",
            "font.size": 10,
            "axes.titlesize": 11,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "legend.title_fontsize": 8.5,
            "axes.linewidth": 0.8,
            "grid.linewidth": 0.55,
            "savefig.dpi": 600,
            "figure.dpi": 150,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def save_figure(fig: plt.Figure, output_dir: Path, stem: str) -> None:
    temporary_png = output_dir / f"{stem}.tmp.png"
    fig.savefig(temporary_png, dpi=600, bbox_inches="tight", facecolor="white")
    temporary_png.replace(output_dir / f"{stem}.png")
    fig.savefig(output_dir / f"{stem}.pdf", bbox_inches="tight", facecolor="white")
    plt.close(fig)


def figure_multiseed(all_runs: pd.DataFrame, aggregate: pd.DataFrame, output_dir: Path) -> None:
    order = aggregate.sort_values("validation_rank")["architecture"].tolist()
    family_by_model = aggregate.set_index("architecture")["family"].to_dict()
    y_positions = np.arange(len(order))
    seed_offsets = {seed: offset for seed, offset in zip(sorted(all_runs["seed"].unique()), [-0.13, 0.0, 0.13])}

    fig, axes = plt.subplots(1, 2, figsize=(7.35, 4.6), sharey=True, constrained_layout=True)
    panels = [
        ("val_f1_macro", "a) Validación (selección)"),
        ("test_f1_macro", "b) Prueba (generalización)"),
    ]

    for ax, (metric, title) in zip(axes, panels):
        selected_y = order.index(SELECTED)
        ax.axhspan(selected_y - 0.42, selected_y + 0.42, color=HIGHLIGHT, alpha=0.16, zorder=0)

        for y, architecture in zip(y_positions, order):
            family = family_by_model[architecture]
            color = PALETTE[family]
            values = all_runs.loc[all_runs["architecture"] == architecture, ["seed", metric]].sort_values("seed")
            mean = values[metric].mean()
            std = values[metric].std(ddof=1)

            for _, row in values.iterrows():
                ax.scatter(
                    row[metric], y + seed_offsets[row["seed"]],
                    s=18, color=color, alpha=0.70, edgecolor="white", linewidth=0.45, zorder=3,
                )

            ax.errorbar(
                mean, y, xerr=std, fmt=MARKERS[family], markersize=6.2,
                markerfacecolor=color, markeredgecolor="#1A1A1A", markeredgewidth=0.65,
                ecolor="#333333", elinewidth=1.0, capsize=2.8, capthick=1.0, zorder=4,
            )

        ax.set_title(title, loc="left", fontweight="semibold", pad=7)
        ax.set_xlabel("F1 macro")
        ax.set_xlim(0.72, 1.008)
        ax.set_xticks(np.arange(0.75, 1.001, 0.05))
        ax.xaxis.set_major_formatter(decimal_comma(2))
        ax.xaxis.grid(True, color="#D7D7D7", linestyle="--", alpha=0.8)
        ax.yaxis.grid(False)
        sns.despine(ax=ax)

    labels = [f"{name} (seleccionado)" if name == SELECTED else name for name in order]
    axes[0].set_yticks(y_positions, labels)
    axes[0].get_yticklabels()[order.index(SELECTED)].set_fontweight("bold")
    axes[0].get_yticklabels()[order.index(SELECTED)].set_color("#B46900")
    axes[0].set_ylabel("Arquitectura (ordenada por validación)")
    axes[0].invert_yaxis()

    legend_handles = [
        Line2D([0], [0], marker=MARKERS[family], color="none", label=("ResNet-50" if family == "ResNet" else family),
               markerfacecolor=PALETTE[family], markeredgecolor="#1A1A1A", markersize=6.5)
        for family in ["YOLO11", "YOLO26", "ResNet"]
    ]
    legend_handles += [
        Line2D([0], [0], marker="o", color="#333333", label="Media ± DE; puntos = semillas",
               markerfacecolor="white", markeredgecolor="#333333", markersize=5),
        Patch(facecolor=HIGHLIGHT, edgecolor="none", alpha=0.35, label="Modelo seleccionado"),
    ]
    fig.legend(handles=legend_handles, loc="outside lower center", ncol=3, frameon=False, columnspacing=1.4)
    fig.get_layout_engine().set(h_pad=0.06, w_pad=0.08, hspace=0.02, wspace=0.05, rect=(0, 0.10, 1, 1))
    save_figure(fig, output_dir, "Figura_NB10_F1_multisemilla")


def figure_class_heatmap(class_means: pd.DataFrame, aggregate: pd.DataFrame, output_dir: Path) -> None:
    order = aggregate.sort_values("validation_rank")["architecture"].tolist()
    matrix = class_means.set_index("architecture").reindex(order)
    matrix = matrix[["damaged", "empty", "overripe", "ripe", "unripe"]]
    matrix.columns = ["Dañado", "Vacío", "Sobremaduro", "Maduro", "Inmaduro"]

    fig, ax = plt.subplots(figsize=(7.35, 4.55), constrained_layout=True)
    heat = sns.heatmap(
        matrix,
        annot=True,
        fmt=".3f",
        cmap="cividis",
        vmin=0.65,
        vmax=1.00,
        linewidths=0.8,
        linecolor="white",
        cbar_kws={"label": "F1 por clase", "shrink": 0.86, "pad": 0.025},
        annot_kws={"fontsize": 9},
        ax=ax,
    )
    for text, value in zip(heat.texts, matrix.to_numpy().ravel()):
        text.set_color("white" if value < 0.86 else "#111111")
        text.set_fontweight("semibold")

    selected_row = order.index(SELECTED)
    ax.add_patch(Rectangle((0, selected_row), matrix.shape[1], 1, fill=False,
                           edgecolor="#E69F00", linewidth=2.2, clip_on=False))
    y_labels = [f"{name} (seleccionado)" if name == SELECTED else name for name in order]
    ax.set_yticklabels(y_labels, rotation=0)
    ax.get_yticklabels()[selected_row].set_fontweight("bold")
    ax.get_yticklabels()[selected_row].set_color("#B46900")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_xlabel("Clase de madurez o condición")
    ax.set_ylabel("Arquitectura (ordenada por validación)")
    ax.set_title("F1 por clase en el conjunto de prueba", loc="left", fontweight="semibold", pad=8)
    cbar = heat.collections[0].colorbar
    cbar.set_ticks([0.65, 0.70, 0.80, 0.90, 1.00])
    cbar.ax.yaxis.set_major_formatter(decimal_comma(2))
    fig.text(0.01, 0.005, "Promedio de tres semillas; escala cromática ampliada de 0,65 a 1,00.",
             fontsize=8, color="#444444")
    fig.get_layout_engine().set(rect=(0, 0.035, 1, 1))
    save_figure(fig, output_dir, "Figura_NB10_F1_por_clase")


def _bubble_size(model_size_mb: float) -> float:
    return 62 + 82 * np.log10(1 + model_size_mb)


def figure_speed(aggregate: pd.DataFrame, output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(7.35, 4.65), constrained_layout=True)
    texts = []

    # Non-dominated models: lower latency and higher F1 are preferred.
    frontier = []
    for _, row in aggregate.iterrows():
        dominated = ((aggregate["inference_ms_image_mean"] <= row["inference_ms_image_mean"]) &
                     (aggregate["test_f1_macro_mean"] >= row["test_f1_macro_mean"]) &
                     ((aggregate["inference_ms_image_mean"] < row["inference_ms_image_mean"]) |
                      (aggregate["test_f1_macro_mean"] > row["test_f1_macro_mean"]))).any()
        if not dominated:
            frontier.append(row)
    frontier_df = pd.DataFrame(frontier).sort_values("inference_ms_image_mean")
    ax.plot(frontier_df["inference_ms_image_mean"], frontier_df["test_f1_macro_mean"],
            linestyle="--", linewidth=1.1, color="#666666", alpha=0.75, zorder=1)

    for _, row in aggregate.iterrows():
        family = row["family"]
        x = row["inference_ms_image_mean"]
        y = row["test_f1_macro_mean"]
        size = _bubble_size(row["model_size_mb_mean"])
        if row["architecture"] == SELECTED:
            ax.scatter(x, y, s=size + 125, facecolor="none", edgecolor="#E69F00",
                       linewidth=2.0, zorder=3)
        ax.scatter(
            x, y, s=size, marker=MARKERS[family], color=PALETTE[family],
            edgecolor="#1A1A1A", linewidth=0.75, zorder=4,
        )
        texts.append(ax.text(x, y, row["architecture"], fontsize=8.5,
                             bbox={"boxstyle": "round,pad=0.16", "fc": "white", "ec": "none", "alpha": 0.86}))

    adjust_text(
        texts,
        ax=ax,
        expand=(1.18, 1.25),
        arrowprops={"arrowstyle": "-", "color": "#777777", "lw": 0.55},
    )
    ax.set_xlabel("Latencia media (ms por imagen)")
    ax.set_ylabel("F1 macro medio en prueba")
    ax.set_title("Rendimiento frente a velocidad de inferencia", loc="left", fontweight="semibold", pad=8)
    ax.set_xlim(6.8, 17.45)
    ax.set_ylim(0.81, 0.935)
    ax.yaxis.set_major_formatter(decimal_comma(2))
    ax.grid(True, color="#D7D7D7", linestyle="--", alpha=0.75)
    sns.despine(ax=ax)

    family_legend = [
        Line2D([0], [0], marker=MARKERS[family], color="none", label=("ResNet-50" if family == "ResNet" else family),
               markerfacecolor=PALETTE[family], markeredgecolor="#1A1A1A", markersize=6.8)
        for family in ["YOLO11", "YOLO26", "ResNet"]
    ]
    family_legend += [
        Line2D([0], [0], color="#666666", linestyle="--", linewidth=1.1, label="Frontera no dominada"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="none", markeredgecolor="#E69F00",
               markeredgewidth=1.6, markersize=9, label="Modelo seleccionado"),
    ]
    ax.legend(handles=family_legend, loc="lower left", ncol=2, frameon=True, framealpha=0.95,
              edgecolor="#D0D0D0", columnspacing=1.1, handletextpad=0.55)
    fig.text(0.995, 0.005, "El área del marcador representa el tamaño medio del modelo.",
             ha="right", fontsize=8, color="#444444")
    fig.get_layout_engine().set(rect=(0, 0.035, 1, 1))
    save_figure(fig, output_dir, "Figura_NB10_F1_velocidad")


# La celda puede ejecutarse de manera independiente después de instalar las dependencias.
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
NB10_RESULTS = PROJECT_ROOT / '06_RESULTADOS' / 'NB10_RESNET50_BENCHMARK'
required_files = {
    'corridas': NB10_RESULTS / 'benchmark_YOLO_ResNet50_por_corrida.csv',
    'clases': NB10_RESULTS / 'F1_test_por_clase_promedio.csv',
    'resumen': NB10_RESULTS / 'benchmark_resumen_por_modelo.csv',
}
missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Faltan resultados de NB10:\n' + '\n'.join(missing))

all_runs = pd.read_csv(required_files['corridas'])
class_means = pd.read_csv(required_files['clases'])
aggregate = pd.read_csv(required_files['resumen'])

configure_style()
figure_multiseed(all_runs, aggregate, NB10_RESULTS)
figure_class_heatmap(class_means, aggregate, NB10_RESULTS)
figure_speed(aggregate, NB10_RESULTS)

print('=' * 72)
print('FIGURAS NB10 ACTUALIZADAS PARA EL CAPÍTULO 7')
print('PNG: 600 dpi | PDF: formato vectorial')
print('Resultados:', NB10_RESULTS)
print('=' * 72)


## 7. Selección y cierre


In [ ]:
winner = aggregate.iloc[0]
selection = {
    'selection_rule': 'Mayor F1 macro promedio en validación; prueba no utilizada para seleccionar.',
    'selected_model': winner['architecture'],
    'selected_val_f1_macro_mean': float(winner['val_f1_macro_mean']),
    'selected_test_f1_macro_mean': float(winner['test_f1_macro_mean']),
    'seeds': SEEDS,
    'comparison_scope': 'YOLO11, YOLO26 y ResNet-50 sobre el mismo split.',
    'warning': 'Las resoluciones y recetas se adaptan a cada familia; interpretar precisión y coste conjuntamente.',
}
(NB10_RESULTS / 'seleccion_modelo.json').write_text(
    json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8'
)

environment = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'scikit_learn': sklearn.__version__,
    'gpu': GPU_NAME,
    'gpu_memory_gb': round(GPU_MEMORY_GB, 2),
    'models': ['YOLO11 n/s/m', 'YOLO26 n/s/m', 'ResNet-50'],
    'seeds': SEEDS,
}
(NB10_RESULTS / 'entorno_ejecucion.json').write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('=' * 72)
print('NB10 FINALIZADO CORRECTAMENTE')
print('Modelo seleccionado por validación:', winner['architecture'])
print('Resultados:', NB10_RESULTS)
print('=' * 72)


## Resultado esperado

El entrenamiento completo termina con **`NB10 FINALIZADO CORRECTAMENTE`**.  
Para actualizar únicamente las figuras, ejecute la celda de instalación y la sección 6; debe aparecer **`FIGURAS NB10 ACTUALIZADAS PARA EL CAPÍTULO 7`**.
